In [1]:
import sys
print(sys.version)

3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]


In [2]:
import os

os.makedirs("documents", exist_ok=True)
os.makedirs("chroma_db", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [3]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'chroma_db', 'documents', 'rag_chatbot.ipynb']


In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("documents/sample.pdf")

docs = loader.load()

print("Number of pages:", len(docs))

C:\Users\91961\AppData\Local\Temp\ipykernel_34324\990292727.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\91961\anaconda3\anaconda\envs\ragbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of pages: 73


In [5]:
print(docs[0].page_content[:1000])

EARTHQUAKE


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))

Number of chunks: 16


In [7]:
print(chunks[0].page_content)

EARTHQUAKE


In [8]:
print(chunks[1].page_content)

Causes of Earthquakes
• The Earth’s crust consists of seven large 
lithospheric plates and numerous 
smaller plates.
• These plates move towards each other 
(a convergent boundary), apart (a 
divergent boundary) or past each other 
(a transform boundary).


In [9]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embeddings model loaded!")

Embeddings model loaded!


C:\Users\91961\AppData\Local\Temp\ipykernel_34324\469454499.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


In [10]:
vector = embeddings.embed_query(
    "What is artificial intelligence?"
)

print("Vector length:", len(vector))

Vector length: 768


In [11]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("Vector database created!")

Vector database created!


In [12]:
query = "What is this document about?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:500])


--- Result 1 ---
What is Magnitude?
Definition:
Magnitude is a measure of size that is often used in geology to describe the size 
of an earthquake and is also believed to be a measure of the amount of energy 
that is released when an earthquake takes place.
How it is measured:
With earthquakes, there is a lot of movement that occurs. Magnitude is often 
measured as the greatest amount of movement that takes place or the largest 
amount of fault area that has moved.
Scales used for measurements:

--- Result 2 ---
What is an Earthquake or How Earthquake Occurs??
 Earthquakes are caused by a sudden release of stress along faults in the earth's crust. The continuous 
motion of tectonic plates causes a steady build-up of pressure in the rock strata on both sides of a fault until the 
stress is sufficiently great that it is released in a sudden, jerky movement. The resulting waves of seismic energy 
propagate through the ground and over its surface, causing the shaking we perceive as earth

In [13]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="deepseek-r1:8b"
)

print("LLM loaded!")

LLM loaded!


C:\Users\91961\AppData\Local\Temp\ipykernel_34324\964260285.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [14]:
response = llm.invoke(
    "What is an earthquake? Answer in 2 sentences."
)

print(response)

An earthquake is the sudden shaking of the ground caused by the release of energy from the Earth's crust. This energy release typically occurs due to the movement of tectonic plates or volcanic activity along fault lines.


In [15]:
import os

os.makedirs("src", exist_ok=True)
os.makedirs("documents", exist_ok=True)
os.makedirs("chroma_db", exist_ok=True)

In [16]:
from src.pdf_loader import load_pdf

docs = load_pdf("documents/sample.pdf")

print(len(docs))

73


In [17]:
from src.pdf_loader import load_pdf
from src.vector_store import build_vectorstore

docs = load_pdf("documents/sample.pdf")

vectorstore = build_vectorstore(docs)

print("Vector store ready!")

Vector store ready!


In [18]:
from src.retriever import get_retriever

retriever = get_retriever(vectorstore)

docs = retriever.invoke(
    "What is magnitude?"
)

print(len(docs))

5


In [19]:
for doc in docs:
    print(doc.page_content[:300])
    print("=" * 50)

Scales used for measurements:
Various scales have been used including the Richter magnitude, also known as 
the local magnitude (ML); and moment magnitude (Mw). The Mw is calculated 
based on how big the area of a fault is that has ruptured.
What is Magnitude?
Definition:
Magnitude is a measure of size that is often used in geology to describe the size 
of an earthquake and is also believed to be a measure of the amount of energy 
that is released when an earthquake takes place.
How it is measured:
With earthquakes, there is a lot of mo
What is an Earthquake or How Earthquake Occurs??
 Earthquakes are caused by a sudden release of stress along faults in the earth's crust. The continuous 
motion of tectonic plates causes a steady build-up of pressure in the rock strata on both sides of a fault until the 
stress is sufficiently great
EARTHQUAKE
What to Do After an Earthquake
•Check yourself and others for injuries. Provide first aid for anyone who needs it.
•Check water, gas, and electri

In [20]:
from src.llm import load_llm

llm = load_llm()

response = llm.invoke(
    "What is an earthquake?"
)

print(response)

An **earthquake** is the sudden and often violent shaking of the ground, caused by the movement of tectonic plates or other geological processes deep beneath the Earth's surface.

Here's a breakdown of the key points:

1.  **Cause:** Earthquakes are primarily caused by the movement of massive slabs of rock called **tectonic plates** that make up the Earth's outer layer (the lithosphere). These plates constantly move, albeit very slowly. When the stress along the boundaries where these plates meet builds up and exceeds the strength of the rocks, they suddenly slip, releasing a tremendous amount of stored energy. This is often compared to a rock breaking under stress (the "elastic rebound theory").
2.  **Focus (Hypocenter):** This is the exact point *underground* where the earthquake starts. The energy is released along a fault line – a fracture or zone of fractures between two blocks of rock.
3.  **Epicenter:** This is the point on the Earth's surface directly above the focus (hypocente

In [21]:
from src.rag_chain import ask_question

answer, sources = ask_question(
    "What is magnitude?",
    retriever,
    llm
)

print(answer)

Magnitude is a measure of size that is often used in geology to describe the size of an earthquake and is also believed to be a measure of the amount of energy that is released when an earthquake takes place.
